<a href="https://colab.research.google.com/github/ARCILAEDNA/SPRINT-9-PROYECTO/blob/main/S9_Version_Student_Proyecto_Landing_Experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experimento A/B en página de inicio

El objetivo de este proyecto es evaluar un **experimento A/B** realizado en una página de inicio (landing page) con versiónes **A y B** para apoyar una **decisión de negocio basada en datos**.

---

El archivo `landing_experiment.csv` contiene información de usuarios expuestos a dos versiones de la página de inicio (landing page) dentro del experimento A/B. Incluye región, dispositivo, fuente de tráfico, tipo de usuario, conversión y gasto.

El análisis sigue una lógica clara y progresiva:

1. 🔍 Explorar y validar los datos.

2. 💰 Comparar el **gasto promedio** por usuario entre la página A y B.

3. 🎯 Comparar la **tasa de conversión** entre la página A y B.

4. 🌐 Revisar **la relación entre la fuente de tráfico y la conversión**.

5. 👤 Revisar **la relación entre el tipo de usuario y la conversión**.

6. 📈 **Visualizar los resultados**: Respalda tus conclusiones mediante gráficos claros.

Se aplican **puebas estadísticas apropiados** para comparar las páginas y **recomendar qué versión es mejor**, justificando la decisión con datos.

## 🧩 Paso 1: Cargar y validar los datos

### 1.1 Carga de datos y vista rápida

In [3]:
# importar librerías
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import ttest_ind
from scipy.stats import chi2_contingency
from scipy.stats import levene
from statsmodels.stats.proportion import proportions_ztest

In [4]:
# cargar archivo
df = pd.read_csv('landing_experiment.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'landing_experiment.csv'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**Vista previa e información general del conjunto de datos**

In [5]:
# mostrar las primeras 5 filas
df.head(5)

NameError: name 'df' is not defined

In [ ]:
# información general
df.info()

✍️ **Comentario**:

    - El dataset no tiene valores ausentes.
    - La columna date es de tipo de dato **object** y debería de ser **datetime**.
   

**Descripción del conjunto de datos**

El dataset contiene las siguientes columnas:

- `user_id` — Identificador único del usuario
- `date` — Fecha en la que el usuario fue expuesto a la página
- `landing` — Versión de la página mostrada al usuario
- `region` — Región geográfica del usuario
- `dispositivo` — Tipo de dispositivo utilizado por el usuario
- `traffic_source` — Canal por el que llegó el usuario
- `user_type` — Tipo de usuario según historial previo
- `converted` — Indica si el usuario realizó una conversión
- `gasto` — Monto gastado por el usuario (0 si no convirtió)

### 1.2 Análisis exploratorio y revisión de calidad de datos

Se identifican las variables clave del experimento A/B y se valida que estén bien definidas, completas y que sean consistentes.


 **Variable `user_id`**  
 Verificar usuarios únicos

In [ ]:
df['user_id'].duplicated().sum()

 **Variable `date`**  
Explorar rango de fechas

In [6]:
df["date"]=pd.to_datetime(df["date"])
df.info()

NameError: name 'df' is not defined

In [ ]:
# Resumen estadístico
df["date"].describe()

In [7]:
# Identificar rango temporal del experimento
print("Fecha mínima:", df["date"].min())
print("Fecha máxima:", df["date"].max())

NameError: name 'df' is not defined

**Variable `gasto` (numérica)**

In [8]:
# Resumen estadístico
df["gasto"].describe()

NameError: name 'df' is not defined

In [ ]:
# Resumen estadístico de usuarios que se convirtieron
df[df['converted']==1]['gasto'].describe()

 **Variables categóricas**  
 Verificar categorías esperadas del experimento ( A y B).

In [9]:
# Explorar variables categóricas y cómo se distribuyen
print("\nConteo de categorías:")
print(df['landing'].value_counts())
print()
print(df['region'].value_counts())
print()
print(df['dispositivo'].value_counts())
print()
print(df['traffic_source'].value_counts())
print()
print(df['user_type'].value_counts())


Conteo de categorías:


NameError: name 'df' is not defined

✍️ **Comentario**:
   - Todas las columnas tienen valores esperados.

## 💰 Paso 2: Comparar el gasto promedio por usuario (página A vs B)

Se evalua si existen diferencias estadísticamente significativas en el gasto promedio de los **usuarios que se convirtieron en clientes** entre la página A y la página B, para identificar qué versión genera **mayor valor económico** para el negocio.


In [ ]:
# Gasto por versión
gasto_A = df[(df['landing']=='A')&(df['converted']==1)]['gasto']
gasto_B = df[(df['landing']=='B')&(df['converted']==1)]['gasto']

# Verificar cantidad de datos que tiene cada grupo
len(gasto_A), len(gasto_B)

### Prueba t de Student

**Hipótesis:**
- **Hipótesis nula (H₀):** el gasto promedio entre el grupo A y el grupo B no presenta diferencias
- **Hipótesis alternativa (H₁):** el gasto promedio entre el grupo A y el grupo B presenta diferencias

In [ ]:
#Prueba de Varianzas
l_stat,p_value_var= levene(gasto_A, gasto_B)
print(f"Estadístico de Levene : {l_stat}")
print(f"Valor p: {p_value_var}")

#Umbral de significancia
alpha=0.05
if p_value_var < alpha:
    print("H0 rechazada: varianzas diferentes (equal_var=False)")
else:
    print("H0 no rechazada:no hay evidencia de varianzas diferentes (equal_var=True)")

In [ ]:
# Aplicar prueba t con varianza= False
t_stat, p_value= ttest_ind(gasto_A, gasto_B, equal_var=False)

# Visualizar resultados
print(f"Estadístico : {t_stat}")
print(f"Valor p: {p_value}")
print()

#Umbral de significancia
alpha=0.05
if p_value < alpha:
    print("H0 rechazada: hay evidencia de diferencia de gasto entre las pâginas A y B")
else:
    print("H0 no rechazada:no hay evidencia suficiente de diferencia de gasto entre las pâginas A y B")

In [10]:
#Calculo de promedios y dirección
media_gastoA = gasto_A.mean()
media_gastoB = gasto_B.mean()

print('media_gastoA', media_gastoA)
print('media_gastoB', media_gastoB)

NameError: name 'gasto_A' is not defined


### 📝 Conclusión e interpretación

**Decisión:**  
Se rechaza la hipótesis nula

**Interpretación de negocio:**  
Existe una diferencia estadísticamente significativa del gasto promedio entre la página A y la página B, es decir, que dicha diferencia no es fruto del azar. En promedio, los usuarios que convirtieron en la página B, gastaron más que los de la página A. El resultado no evalúa si la diferencia observada es suficiente para justificar un cambio desde una perspectiva  de negocio.

---


## 📈 Paso 3: Comparar la tasa de conversión entre la página A y B

Se evalua si existen diferencias estadísticamente significativas en la **tasa de conversión** entre la página A y B, con el fin de identificar qué versión genera **mayor número de usuarios convertidos**.

### Prueba z

**Hipótesis:**
- **Hipótesis nula (H₀):** La tasa de conversión de la página A es igual a la de la página B
- **Hipótesis alternativa (H₁):**  La tasa de conversión de la página A es diferente a la de la página B

In [11]:
# Número de usuarios convertidos por página
df_filtrado= df[df['converted']==1]
conversiones= df_filtrado.groupby('landing')['converted'].count()

# Total de usuarios por página
totales= df.groupby('landing')['converted'].count()

print("Usuarios convertidos por página:\n", conversiones)
print("\nTotal de usuarios por página:\n", totales)

NameError: name 'df' is not defined

In [12]:
#Valores en formato lista:
exitos=[conversiones['A'], conversiones ['B']]
observaciones= [totales['A'], totales ['B']]

# Aplicar prueba z
z_stat, p_value = proportions_ztest (exitos, observaciones)

# Visualizar resultados
print(f"Estadístico : {z_stat}")
print(f"Valor p: {p_value}")
print()

#Umbral de significancia
alpha=0.05
if p_value < alpha:
    print("H0 rechazada: La tasa de conversión de la página A es diferente a la de la página B")
else:
    print("H0 no rechazada: La tasa de conversión de la página A es igual a la de la página B")


NameError: name 'conversiones' is not defined

In [13]:
#Comparando las tasas de conversiôn entre pâginas A y B
tasa_A= exitos[0]/observaciones[0]
tasa_B= exitos[1]/observaciones[1]

print(f"tasa de conversión página A:{tasa_A:.2%}")
print(f"tasa de conversión página B:{tasa_B:.2%}")

NameError: name 'exitos' is not defined

### 📝 Conclusión e interpretación

**Decisión:**  
Se rechaza la hipótesis nula.

**Interpretación de negocio:**  
Con base en una prueba z para proporciones, encoontramos evidencia estadística de una diferencia en la tasa de conversión entre las páginas A y B. En particular, la pâgina B muestra una tasa de conversiôn mayor que la pâgina A. Este resultado no evalúa si la magnitud de la diferencia es suficiente para justificar una decisión de negocio.

## 🔗 Paso 4: Revisar la relación entre la fuente de tráfico y la conversión

Se analiza si existe una **asociación estadísticamente significativa** entre la **fuente de tráfico** (`traffic_source`) y la **conversión** (`converted`), para identificar qué canales generan más conversiones.

### Prueba chi cuadrada para variables categôricas

**Hipótesis:**
- **Hipótesis nula (H₀):** La conversión es independiente del canal por el que llegó el usuario
- **Hipótesis alternativa (H₁):** La conversión es dependiente del canal por el que llegó el usuario

In [ ]:
# conteo de valores
df['converted'].value_counts()
df[['traffic_source', 'converted']].value_counts()
#tabla
tabla=pd.crosstab(df['traffic_source'], df['converted'])
tabla

In [14]:
# Aplicar prueba
chi2_contingency(tabla)
chi2_stat, p_value, dof, expected= chi2_contingency(tabla)

print(f"Estadîstico chi-cuadrado: {chi2_stat:.3f} ")
print(f"Valor P: {p_value:.3f}")
print(f"Grados de libertad: {dof}")
print("\nFrecuencias esperadas:")
print(expected)

#Umbral de significancia
alpha=0.05
if p_value < alpha:
    print("H0 rechazada: hay evidencia de asociaciôn entre las variables")
else:
    print("H0 no rechazada: no hay evidencia suficiente de asociaciôn entre las variables")


NameError: name 'tabla' is not defined

In [ ]:
# Tabla de contingencia normalizada
tabla_traffic_source = pd.crosstab(df['traffic_source'], df['converted'], normalize='index')*100
tabla_traffic_source

### 📝 Conclusión e interpretación

**Decisión:**  
Se rechaza la hipótesis nula.

**Interpretación de negocio:**  
Con base en la prueba de chi cuadrado de independencia, existe evidencia estadística de que la conversión está asociada al canal de tráfico. En particular, los usuarios que llegan por Email presentan una tasa de conversión mayor que los que llegan por tráfico orgánico. Este resultado no implica causalidad directa ni evalúa si esta diferencia justifica decisiones diferenciadas desde una perspectiva de negocio.

## 👤 Paso 5: Revisar la relación entre el tipo de usuario y la conversión

Se analiza si existe una **asociación estadísticamente significativa** entre el **tipo de usuario** (`user_type`) y la **conversión** (`converted`), entendiendo que un usuario recurrente puede haber visitado antes sin necesariamente convertirse en cliente en esta ocasión.

El objetivo es identificar qué perfiles muestran mayor probabilidad de conversión dentro del contexto analizado.


### Prueba chi cuadrada para variables categôricas

**Hipótesis:**
- **Hipótesis nula (H₀):** La conversión es independiente del tipo de usuario
- **Hipótesis alternativa (H₁):** La conversión es dependiente del tipo de usuario


In [ ]:
# conteo de valores
df['converted'].value_counts()
df[['user_type', 'converted']].value_counts()
#tabla
tabla_1=pd.crosstab(df['user_type'], df['converted'])
tabla_1

In [ ]:
# Aplicar prueba
chi2_contingency(tabla_1)
chi2_stat, p_value, dof, expected= chi2_contingency(tabla_1)

print(f"Estadîstico chi-cuadrado: {chi2_stat:.3f} ")
print(f"Valor P: {p_value:.3f}")
print(f"Grados de libertad: {dof}")
print("\nFrecuencias esperadas:")
print(expected)

#Umbral de significancia
alpha=0.05
if p_value < alpha:
    print("H0 rechazada: hay evidencia de asociación entre las variables")
else:
    print("H0 no rechazada: no hay evidencia suficiente de asociación entre las variables")


In [ ]:
# Tabla de contingencia normalizada
tabla_user_type= pd.crosstab(df['user_type'], df['converted'], normalize='index')*100
tabla_user_type

### 📝 Conclusión e interpretación

**Decisión:**  
No se rechaza la hipótesis nula

**Interpretación de negocio:**  
Con base en la prueba de chi cuadrado de independencia, no existe evidencia estadística suficiente de que la conversión está asociada al tipo de usuario. Este resultado no implica causalidad directa ni evalúa si esta diferencia justifica decisiones diferenciadas desde una perspectiva de negocio.

## 📊 Paso 6: Visualizar los resultados de variables categóricas

Se explora visualmente la relación entre variables categóricas (`traffic_source` y `user_type`) y la conversión, mostrando para cada categoría:
- la cantidad absoluta de usuarios que convirtieron y no convirtieron,
- la proporción de usuarios que convirtieron y no convirtieron.

Esto permite analizar tanto el impacto en volumen como la efectividad relativa de cada categoría y reforzar los resultados obtenidos en las pruebas estadísticas.

### Relación entre la fuente de tráfico y la conversión

In [15]:
#Gráfico de barras agrupadas
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='traffic_source', hue='converted', palette= ["#95D5B2", "#F4A261"])

plt.title('Conversiones por fuente de trâfico', fontsize=15)
plt.xlabel('traffic_source', fontsize=12)
plt.ylabel('cantidad de usuarios', fontsize=12)
plt.legend(title='Convirtió?', labels=['No(0)', 'Si(1)'])
plt.show()

NameError: name 'df' is not defined

<Figure size 1000x500 with 0 Axes>

In [ ]:
#Gráfico de barras apiladas
tabla_traffic_source.plot(kind='bar', stacked=True, figsize=(10,6), color=["#8ecae6", "#95d5b2"])

plt.title('Tasa de conversiôn por fuente de trâfico', fontsize=15)
plt.ylabel('porcentaje(%)', fontsize=12)
plt.xlabel('traffic_source', fontsize=12)
plt.legend(title='Convirtió?', labels=['No(0)', 'Si(1)'], loc='upper left', bbox_to_anchor=(1,1))
plt.show()

**Conversiones por fuente de tráfico**
El gráfico muestra la cantidad de usuarios que convirtieron y que no convirtieron según su fuente de tráfico. Organic es el canal que concentra el mayor volumen de usuarios (17.987 en total), seguido por Ads (11.935), Email (6.123) y Referral (3.955). En todos los canales los usuarios que no convierten superan ampliamente a los que sí lo hacen, lo que es coherente con las tasas de conversión globales calculadas previamente. Como el volumen de tráfico difiere entre canales, el número absoluto de conversiones es mayor en Organic y Ads, pero esto refleja principalmente su tamaño y no necesariamente un mejor desempeño.

**Tasa de conversión por fuente de tráfico**
Las cuatro fuentes presentan proporciones muy similares: Email registra la tasa de conversión más alta (14,99 %), seguido por Ads (14,74 %), Referral (13,88 %) y Organic (13,79 %). La diferencia entre el mejor y el peor canal es de apenas 1,2 puntos porcentuales, y en todos los casos entre el 85,0 % y el 86,2 % de los usuarios no convierte. En términos prácticos, los canales Email y Ads tienden a convertir ligeramente mejor que Organic y Referral, pero la fuente de tráfico explica una parte mínima de la conversión, por lo que probablemente otras variables tengan mayor peso.

### Relación entre el tipo de usuario y la conversión

In [ ]:
#Gráfico de barras agrupadas
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='user_type', hue='converted', palette= ["#95D5B2", "#F4A261"])

plt.title('Conversiones por tipo de usuario', fontsize=15)
plt.xlabel('user_type', fontsize=12)
plt.ylabel('cantidad de usuarios', fontsize=12)
plt.legend(title='Convirtió?', labels=['No(0)', 'Si(1)'])
plt.show()

In [ ]:
#Gráfico de barras apiladas
tabla_user_type.plot(kind='bar', stacked=True, figsize=(10,6), color=["#8ecae6", "#95d5b2"])

plt.title('Tasa de conversiôn por tipo de usuario', fontsize=15)
plt.ylabel('porcentaje(%)', fontsize=12)
plt.xlabel('user_type', fontsize=12)
plt.legend(title='Convirtió?', labels=['No(0)', 'Si(1)'], loc='upper left', bbox_to_anchor=(1,1))
plt.show()

**Conversiones por tipo de usuario**

El gráfico compara la cantidad de usuarios que convirtieron y que no convirtieron según su tipo. Los usuarios nuevos constituyen el grupo más numeroso (26.033 frente a 13.967 recurrentes), por lo que concentran también un mayor número absoluto de conversiones. En ambos grupos los usuarios que no convierten superan ampliamente a quienes sí lo hacen, lo que es coherente con las tasas globales calculadas previamente. Como los grupos tienen tamaños distintos, la diferencia en conversiones absolutas refleja principalmente el volumen de cada grupo y no necesariamente un mejor desempeño.

**Tasa de conversión por tipo de usuario**

Al normalizar los datos a porcentaje, se elimina el efecto del tamaño de cada grupo y se obtiene una comparación directa. Los usuarios nuevos presentan una tasa de conversión de 14,36 %, mientras que los recurrentes alcanzan 14,09 %, una diferencia de apenas 0,27 puntos porcentuales. Las dos barras son prácticamente idénticas, y sugiere que el tipo de usuario, por sí solo, no influye de forma apreciable en la probabilidad de convertir.

## 🧩 Paso 7. Insight Ejecutivo para Stakeholders

Se traducen los hallazgos del análisis del experimento A/B en conclusiones accionables para el negocio, enfocadas en **versión de página, conversión, gasto promedio, canales de tráfico y tipo de usuario**.

**Preguntas a responder:**  
- ¿Qué página genera mayor conversión y gasto promedio?  
- ¿Qué canales de tráfico son más efectivos para generar conversiones?  
- ¿Existen diferencias significativas según el tipo de usuario?  
- ¿Qué recomendaciones se pueden tomar para optimizar la estrategia de marketing?


---

### 🌟 Insight Ejecutivo basado en el Experimento A/B

#### 🔍 **Comparación de página (A vs B)**  

###Gasto promedio por usuario que convirtió:

La página B registra un gasto promedio de 68,75 frente a 61,09 de la A (p = 3,6 × 10⁻²¹).

**Tasa de conversión:**
La página B convierte 15,96 % frente a 12,57 % de la A (p = 3,8 × 10⁻²²).

- **Interpretación:**
La diferencia de gasto entre las páginas es estadísticamente significativa y no atribuible al azar. La página B no solo atrae a más compradores, sino que quienes compran en ella gastan más. Esta comparación se limita a usuarios que ya convirtieron, de modo que describe el valor de cada cliente y no el efecto de la página sobre el total de visitantes.
---

#### 📊 **Segmentación por fuente de tráfico**

Observación: Email (14,99 %) y Ads (14,74 %) convierten más que Referral (13,88 %) y Organic (13,79 %); chi-cuadrado = 8,662, p = 0,034.

- **Interpretación:**
Hay asociación significativa, pero el efecto es muy pequeño (diferencia de 1,2 puntos porcentuales).
 ---

#### 📊 **Segmentación por tipo de usuario**

Observación: Nuevos 14,36 % y recurrentes 14,09 %; chi-cuadrado = 0,513, p = 0,474.

- **Interpretación:**
No hay evidencia de asociación entre el tipo de usuario y la conversión.
---

Las visualizaciones usadas respaldan los resultados estadísticos de pasos anteriores.

---

#### 💡 **Recomendaciones de negocio:**

Recomendación 1: Implementar la página B, que supera a la A en conversión y en gasto promedio.

Recomendación 2: Priorizar moderadamente Email y Ads, considerando su costo de adquisición, y no diferenciar la estrategia por tipo de usuario.